## পরিচয় 

এই পাঠে আলোচিত হবে: 
- ফাংশন কলিং কী এবং এর ব্যবহারের ক্ষেত্রসমূহ 
- Azure OpenAI ব্যবহার করে কিভাবে একটি ফাংশন কল তৈরি করবেন 
- কিভাবে একটি ফাংশন কলকে একটি অ্যাপ্লিকেশনেই ইন্টিগ্রেট করবেন 

## শেখার লক্ষ্য 

এই পাঠ শেষ করার পর আপনি জানতে পারবেন এবং বুঝতে পারবেন: 

- ফাংশন কলিং ব্যবহারের উদ্দেশ্য 
- Azure OpenAI সার্ভিস ব্যবহার করে ফাংশন কল সেটআপ 
- আপনার অ্যাপ্লিকেশনের ব্যবহারের জন্য কার্যকরী ফাংশন কল ডিজাইন করা 


## ফাংশন কল বোঝা 

এই পাঠের জন্য, আমরা আমাদের শিক্ষা স্টার্টআপের জন্য একটি ফিচার তৈরি করতে চাই যা ব্যবহারকারীদের একটি চ্যাটবট ব্যবহার করে প্রযুক্তিগত কোর্স খুঁজে পেতে সক্ষম করে। আমরা এমন কোর্সগুলি সুপারিশ করব যা তাদের দক্ষতার স্তর, বর্তমান ভূমিকা এবং আগ্রহের প্রযুক্তির সাথে মানানসই। 

এটি সম্পন্ন করার জন্য আমরা নিম্নলিখিতগুলির সমন্বয় ব্যবহার করব: 
 - ব্যবহারকারীর জন্য একটি চ্যাট অভিজ্ঞতা তৈরি করতে `Azure Open AI`
 - ব্যবহারকারীর অনুরোধের ভিত্তিতে কোর্স খুঁজে পেতে সাহায্য করার জন্য `Microsoft Learn Catalog API`
 - ব্যবহারকারীর প্রশ্ন নেওয়ার জন্য এবং API অনুরোধ তৈরি করতে একটি ফাংশনে পাঠানোর জন্য `Function Calling`

শুরু করতে গেলে, চলুন দেখি আমরা প্রথম স্থানে কেন ফাংশন কল ব্যবহার করতে চাই: 


### কেন ফাংশন কলিং  

যদি আপনি এই কোর্সের অন্য কোনও পাঠ শেষ করে থাকেন, তাহলে সম্ভবত আপনি বড় ভাষার মডেলগুলি (LLMs) ব্যবহার করার শক্তি বুঝতে পেরেছেন। আশা করি আপনি তাদের কিছু সীমাবদ্ধতাও দেখতে পাচ্ছেন। 

ফাংশন কলিং হলো Azure Open AI সার্ভিসের একটি বৈশিষ্ট্য যাতে নিম্নলিখিত সীমাবদ্ধতাগুলি অতিক্রম করা যায়:  
1) সঙ্গতিপূর্ণ প্রতিক্রিয়ার ফরম্যাট  
2) চ্যাট প্রসঙ্গে একটি অ্যাপ্লিকেশনের অন্যান্য উৎস থেকে ডেটা ব্যবহার করার সক্ষমতা  

ফাংশন কলিং এর আগে, LLM থেকে প্রতিক্রিয়াগুলি অপরিষ্কার এবং অসঙ্গতিপূর্ণ ছিল। ডেভেলপারদের প্রতিক্রিয়ার প্রতিটি পরিবর্তন সামাল দিতে জটিল যাচাই কোড লিখতে হতো।  

ব্যবহারকারীরা "স্টকহোমে বর্তমান আবহাওয়া কেমন?" মতো উত্তর পেতেন না। এর কারণ মডেলগুলি শুধুমাত্র প্রশিক্ষণের সময়ের ডেটার মধ্যে সীমাবদ্ধ ছিল।  

নিচের উদাহরণটি দেখে এই সমস্যাটি বোঝা যাক:  

ধরুন আমরা শিক্ষার্থীদের ডেটার একটি ডাটাবেস তৈরি করতে চাই যাতে আমরা তাদেরকে সঠিক কোর্সের পরামর্শ দিতে পারি। নিচে দুইটি শিক্ষার্থীর বর্ণনা রয়েছে যেগুলো তাদের ডেটার দিক থেকে খুব অনুরূপ।  


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


আমরা এটি একটি LLM-এ পাঠাতে চাই যাতে এটি ডেটা পার্স করতে পারে। উপরে পরে এটি আমাদের অ্যাপ্লিকেশনে ব্যবহার করা যেতে পারে একটি API-তে পাঠানোর জন্য বা এটি একটি ডাটাবেসে সংরক্ষণ করার জন্য। 

চলুন দুইটি অভিন্ন প্রম্পট তৈরি করি যেগুলোতে আমরা LLM-কে নির্দেশ দিব কোন তথ্যগুলোতে আমরা আগ্রহী: 


আমরা এটিকে একটি LLM-এ পাঠাতে চাই যাতে এটি আমাদের পণ্যের জন্য গুরুত্বপূর্ণ অংশগুলি পার্স করতে পারে। তাই আমরা LLM-কে নির্দেশ দেওয়ার জন্য দুটি অভিন্ন প্রম্পট তৈরি করতে পারি: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


এই দুইটি প্রম্পট তৈরি করার পরে, আমরা সেগুলো LLM-এ `client.responses.create` ব্যবহার করে পাঠাবো। আমরা প্রম্পটটি `input` ভ্যারিয়েবলে সংরক্ষণ করি এবং রোলটি `user` হিসেবে নির্ধারণ করি। এটি ব্যবহারকারীর বার্তা চ্যাটবট-এ লেখার মতোই নকল করার জন্য। 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

এখন আমরা উভয় অনুরোধ LLM-এ পাঠাতে পারি এবং আমরা যে প্রতিক্রিয়া পেয়েছি তা পরীক্ষা করতে পারি। 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



যদিও প্রম্পটগুলি একই এবং বর্ণনাগুলি মিল রয়েছে, আমরা `Grades` প্রপার্টির বিভিন্ন ফর্ম্যাট পেতে পারি।

উপরের কোষটি একাধিকবার চালালে, ফর্ম্যাট হতে পারে `3.7` অথবা `3.7 GPA`।

এর কারণ হলো LLM লিখিত প্রম্পটের আকারে অসংগঠিত ডেটা গ্রহণ করে এবং অসংগঠিত ডেটাই প্রত্যাবর্তন করে। আমাদের একটি সুশৃঙ্খল ফর্ম্যাট থাকা দরকার যাতে আমরা ডেটা সংরক্ষণ বা ব্যবহার করার সময় কী আশা করতে হবে তা জানতে পারি।

ফাংশনাল কলিং ব্যবহার করে, আমরা নিশ্চিত করতে পারি যে আমরা সুশৃঙ্খল ডেটা ফিরে পাচ্ছি। ফাংশন কলিং ব্যবহার করার সময়, LLM আসলেই কোন ফাংশন কল বা চালায় না। পরিবর্তে, আমরা LLM-এর জন্য একটি কাঠামো তৈরি করি যাতে তার প্রতিক্রিয়াগুলির জন্য অনুসরণ করতে পারে। এরপর আমরা সেগুলি ব্যবহার করি আমাদের অ্যাপ্লিকেশনে কোন ফাংশন চালাতে হবে তা জানতে।
 


![ফাংশন কলিং ফ্লো ডায়াগ্রাম](../../../../translated_images/bn/Function-Flow.083875364af4f4bb.webp)


আমরা তারপর ফাংশন থেকে যা ফিরে আসে তা নিয়ে এটি LLM-এ পাঠাতে পারি। তারপর LLM ব্যবহারকারীর প্রশ্নের উত্তর দিতে প্রাকৃতিক ভাষা ব্যবহার করে প্রতিক্রিয়া জানাবে। 


### ফাংশন কল ব্যবহারের ক্ষেত্রে ব্যবহারের উদাহরণ  

**বাইরের টুলগুলি কল করা**  
চ্যাটবট ব্যবহারকারীদের প্রশ্নের উত্তর দিতে দুর্দান্ত। ফাংশন কল ব্যবহারের মাধ্যমে, চ্যাটবট ব্যবহারকারীর মেসেজ ব্যবহার করে নির্দিষ্ট কাজ সম্পন্ন করতে পারে। উদাহরণস্বরূপ, একজন ছাত্র চ্যাটবটকে বলতে পারে "আমার ইনস্ট্রাক্টরকে ইমেল পাঠাও বলছি যে আমি এই বিষয়ে আরও সহায়তা প্রয়োজন"। এটি `send_email(to: string, body: string)` নামে একটি ফাংশন কল করতে পারে  


**API বা ডাটাবেস অনুসন্ধান তৈরি করা**  
ব্যবহারকারীরা প্রাকৃতিক ভাষা ব্যবহার করে তথ্য খুঁজে পেতে পারেন যা একটি ফরম্যাট করা অনুসন্ধান বা API অনুরোধে রূপান্তরিত হয়। এর একটি উদাহরণ হতে পারে একজন শিক্ষক যিনি প্রশ্ন করেন "শেষ অ্যাসাইনমেন্ট যে ছাত্ররা সম্পন্ন করেছে তারা কারা" যা `get_completed(student_name: string, assignment: int, current_status: string)` নামে একটি ফাংশন কল করতে পারে  


**গঠনমূলক তথ্য তৈরি করা**  
ব্যবহারকারীরা একটি টেক্সট ব্লক বা CSV নিয়ে LLM ব্যবহার করে তার মধ্যে থেকে গুরুত্বপূর্ণ তথ্য বের করতে পারেন। উদাহরণস্বরূপ, একজন ছাত্র শান্তি চুক্তি সম্পর্কিত উইকিপিডিয়া আর্টিকেল ব্যবহার করে AI ফ্ল্যাশ কার্ড তৈরি করতে পারেন। এটি একটি `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` নামে ফাংশন ব্যবহার করে করা যেতে পারে  


## ২. আপনার প্রথম ফাংশন কল তৈরি করা 

ফাংশন কল তৈরি করার প্রক্রিয়ায় ৩টি প্রধান ধাপ রয়েছে: 
১. আপনার ফাংশনের একটি তালিকা এবং একটি ব্যবহারকারীর বার্তা নিয়ে চ্যাট সম্পূর্ণকরণ API কল করা 
২. একটি ফাংশন বা API কল কার্যকর করার জন্য মডেলের প্রতিক্রিয়া পড়ুন 
৩. ব্যবহারকারীর জন্য প্রতিক্রিয়া তৈরি করতে আপনার ফাংশনের প্রতিক্রিয়া নিয়ে চ্যাট সম্পূর্ণকরণ API আরেকটি কল করুন। 


![একটি ফাংশন কলের প্রবাহ](../../../../translated_images/bn/LLM-Flow.3285ed8caf4796d7.webp)


### একটি ফাংশন কলের উপাদানসমূহ 

#### ব্যবহারকারীর ইনপুট 

প্রথম ধাপ হল একটি ব্যবহারকারী বার্তা তৈরি করা। এটি একটি টেক্সট ইনপুটের মান নিয়ে ডায়নামিক্যালি নির্ধারিত হতে পারে অথবা আপনি এখানে একটি মান নির্ধারণ করতে পারেন। যদি এটি আপনার প্রথমবারের মতো Chat Completions API-এর সাথে কাজ করা হয়, তবে আমাদের `role` এবং বার্তার `content` নির্ধারণ করতে হবে। 

`role` হতে পারে `system` (নিয়ম তৈরি করা), `assistant` (মডেল) অথবা `user` (চূড়ান্ত ব্যবহারকারী)। ফাংশন কলিং-এর জন্য, আমরা এটিকে `user` হিসেবে এবং একটি উদাহরণ প্রশ্ন হিসেবে নির্ধারণ করব। 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### ফাংশন তৈরি করা। 

পরবর্তীতে আমরা একটি ফাংশন এবং সেই ফাংশনের প্যারামিটারগুলি সংজ্ঞায়িত করব। আমরা এখানে কেবল একটি ফাংশন ব্যবহার করব যার নাম `search_courses` কিন্তু আপনি একাধিক ফাংশন তৈরি করতে পারেন।

**গুরুত্বপূর্ণ**: ফাংশনগুলি LLM-কে প্রদানকৃত সিস্টেম মেসেজে অন্তর্ভুক্ত থাকে এবং আপনার উপলব্ধ টোকেনের পরিমাণের মধ্যে অন্তর্ভুক্ত থাকবে। 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**সংজ্ঞাসমূহ** 

`name` - সেই ফাংশনের নাম যা আমরা কল করতে চাই। 

`description` - ফাংশনটি কিভাবে কাজ করে তার বর্ণনা। এখানে স্পষ্ট এবং নির্দিষ্ট হওয়া গুরুত্বপূর্ণ 

`parameters` - এমন একটি মান এবং ফরম্যাটের তালিকা যা আপনি চান মডেল তার উত্তরে তৈরি করুক 


`type` - সেই ডেটা টাইপ যেখানে বৈশিষ্ট্যগুলি সংরক্ষণ করা হবে 

`properties` - নির্দিষ্ট মানগুলির তালিকা যা মডেল তার উত্তরে ব্যবহার করবে 


`name` - বৈশিষ্ট্যটির নাম যা মডেল তার ফরম্যাট করা উত্তরে ব্যবহার করবে 

`type` - এই বৈশিষ্ট্যের ডেটা টাইপ 

`description` - নির্দিষ্ট বৈশিষ্ট্যের বর্ণনা 


**ঐচ্ছিক**

`required` - ফাংশন কল সম্পন্ন করার জন্য প্রয়োজনীয় বৈশিষ্ট্য 


### ফাংশন কল করা 
একটি ফাংশন সংজ্ঞায়িত করার পর, এখন আমাদের এটি Chat Completion API-তে কল এ যোগ করতে হবে। আমরা এটি অনুরোধে `functions` যোগ করে করি। এই ক্ষেত্রে `functions=functions`। 

এছাড়াও `function_call` কে `auto` সেট করার অপশন রয়েছে। এর মানে হল আমরা নিজে নির্ধারণ না করে ব্যবহারকারীর মেসেজের ভিত্তিতে কোন ফাংশন কল করা হবে তা LLM ঠিক করবে।


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


এখন আসুন প্রতিক্রিয়াটি দেখি এবং কিভাবে এটি গঠন করা হয়েছে তা দেখি: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

আপনি দেখতে পাচ্ছেন যে ফাংশনের নাম কল করা হয়েছে এবং ব্যবহারকারীর বার্তা থেকে, LLM ফাংশনের আর্গুমেন্টগুলির সাথে মেলানোর জন্য ডেটা খুঁজে পেয়েছে। 


## ৩. ফাংশন কলগুলি একটি অ্যাপ্লিকেশনে সংহত করা। 


LLM থেকে ফরম্যাট করা প্রতিক্রিয়া আমরা পরীক্ষা করার পর, এখন আমরা এটিকে একটি অ্যাপ্লিকেশনে সংহত করতে পারি। 

### প্রবাহ পরিচালনা 

এটিকে আমাদের অ্যাপ্লিকেশনে সংহত করতে, আসুন নিম্নলিখিত পদক্ষেপগুলি গ্রহণ করি: 

প্রথমে, চলুন Open AI সার্ভিসে কল করি এবং বার্তাটি `response_message` নামে একটি ভেরিয়েবলে সংরক্ষণ করি। 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


এখন আমরা একটি ফাংশন সংজ্ঞায়িত করব যা Microsoft Learn API কল করবে একটি কোর্সের তালিকা পাওয়ার জন্য: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



একটি সর্বোত্তম অনুশীলন হিসেবে, আমরা প্রথমে দেখব মডেলটি কি একটি ফাংশন কল করতে চায় কিনা। তার পর, আমরা উপলব্ধ ফাংশনগুলির মধ্যে একটি তৈরি করব এবং সেটিকে কল করা ফাংশনের সাথে মেলাবো।
তারপর আমরা ফাংশনের আর্গুমেন্টগুলো নেব এবং সেগুলোকে LLM-এর আর্গুমেন্টের সাথে ম‍্যাপ করব।

সর্বশেষে, আমরা ফাংশন কল মেসেজ এবং `search_courses` মেসেজ থেকে ফিরে আসা মানগুলো সংযুক্ত করব। এটি LLM-কে সমস্ত তথ্য দেয় যা ব্যবহারকারীর সাথে প্রাকৃতিক ভাষায় উত্তরের জন্য দরকার।
ব্যবহারকারীর সাথে প্রাকৃতিক ভাষায় উত্তর দেওয়ার জন্য।


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


এখন আমরা আপডেট করা বার্তাটি LLM-এ পাঠাবো যাতে আমরা API JSON ফরম্যাটেড প্রতিক্রিয়ার পরিবর্তে একটি প্রাকৃত ভাষার প্রতিক্রিয়া পেতে পারি। 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## কোড চ্যালেঞ্জ 

দুর্দান্ত কাজ! Azure Open AI Function Calling শেখা অব্যাহত রাখতে আপনি তৈরি করতে পারেন: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - ফাংশনের আরও প্যারামিটার যা শিক্ষার্থীদের আরও কোর্স খুঁজে পেতে সাহায্য করতে পারে। আপনি উপলব্ধ API প্যারামিটারগুলি এখানে পেতে পারেন: 
 - আরেকটি ফাংশন কল তৈরি করুন যা শিক্ষার্থীর আরও তথ্য নেয় যেমন তাদের মাতৃভাষা 
 - ফাংশন কল এবং/অথবা API কল যখন উপযুক্ত কোনো কোর্স ফেরত না দেয় তখন ত্রুটি ব্যবস্থাপনা তৈরি করুন 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
